In [1]:
!rm -rf /kagg/working/*

In [2]:
import os
import glob
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import GroupKFold
import warnings
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
from torchinfo import summary

from tqdm.notebook import tqdm
from tabulate import tabulate
from IPython.display import clear_output
from tqdm.notebook import tqdm
from tabulate import tabulate
from IPython.display import clear_output
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

In [3]:
train_label=pd.read_csv("/kaggle/input/competitions/soil-grain-size-from-photos/Training_labels.csv")
train_label.head()

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,F827,9.4904,19.3892,47.6257,89.5554,99.8965,99.9896,100.0000,100.0000,100.0000,100.0000,100.0
1,G190,5.2076,10.2425,23.7537,50.3113,75.8311,85.7233,90.9898,94.8969,98.5814,100.0000,100.0
2,H030,6.7833,10.9091,19.0599,37.3235,66.8349,96.4965,97.9530,99.5691,100.0000,100.0000,100.0
3,H031,1.8556,5.5638,11.4945,19.1022,25.8591,39.6389,50.5450,65.2738,81.4215,94.7783,100.0
4,H037,0.9320,6.4500,16.3216,27.8019,34.6629,46.3652,62.7168,79.0523,89.7830,100.0000,100.0


In [4]:
ppm_csv=pd.read_csv("/kaggle/input/competitions/soil-grain-size-from-photos/ppm.csv")
ppm_csv.head()

,phone,camera,width,height,ppm
0,iPhone 14,iPhone 14,4032,3024,13.942
1,iPhone 16,iPhone 16,5712,4284,19.525
2,Motorola Edge,motorola edge 20,4000,1800,11.492
3,Samsung A52,SM-A525F,9248,6936,26.330


In [5]:
CONFIG = {
    'seed': 42,
    'img_size': 384,
    'batch_size': 16,
    'epochs': 20,
    'lr': 5e-5,
    'min_lr': 1e-6,
    'weight_decay': 1e-4,
    'n_splits': 5,
    'num_workers': 2,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'data_dir': '/kaggle/input/competitions/soil-grain-size-from-photos',
    'work_dir': '/kaggle/working'
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CONFIG['seed'])

In [6]:
class SoilDataset(Dataset):
    def __init__(self, image_paths, ppm_dict, labels_df=None, transform=None):
        self.image_paths = image_paths
        self.ppm_dict = ppm_dict
        self.labels_df = labels_df
        self.transform = transform
        self.mean_ppm = sum(ppm_dict.values()) / len(ppm_dict)
        
        if self.labels_df is not None:
            if 'sample_id' in self.labels_df.columns:
                self.labels_df = self.labels_df.set_index('sample_id')
            self.label_cols = [str(c) for c in [0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2, 6.3, 20, 63, 200]]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        filename = os.path.basename(img_path)
        camera_name = next((cam for cam in self.ppm_dict.keys() if cam in filename), None)
        ppm_val = self.ppm_dict[camera_name] if camera_name else self.mean_ppm
        ppm_tensor = torch.tensor([ppm_val], dtype=torch.float32)
        
        if self.labels_df is not None:
            sample_id = next((vid for vid in self.labels_df.index if vid in filename), None)
            targets = self.labels_df.loc[sample_id, self.label_cols].values.astype(np.float32)
            return image, ppm_tensor, torch.tensor(targets), sample_id
        else:
            match = re.search(r'(TEST_\d+)|([A-Za-z]\d{3})', filename, re.IGNORECASE)
            sample_id = match.group(0).upper() if match else filename.split('_')[0] 
            return image, ppm_tensor, sample_id

train_transforms = T.Compose([
    T.RandomResizedCrop(CONFIG['img_size'], scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transforms = T.Compose([
    T.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [7]:
train_images = np.array(glob.glob(os.path.join(CONFIG['data_dir'], 'Training-All_Photos/*/*.jpg')))
labels_df = pd.read_csv(os.path.join(CONFIG['data_dir'], 'Training_labels.csv'))
ppm_df = pd.read_csv(os.path.join(CONFIG['data_dir'], 'ppm.csv'))

ppm_dict = dict(zip(ppm_df['camera'], ppm_df['ppm']))
ppm_dict.update(dict(zip(ppm_df['phone'], ppm_df['ppm'])))

train_groups = []
for path in train_images:
    filename = os.path.basename(path)
    sid = next((vid for vid in labels_df['sample_id'].values if vid in filename), None)
    train_groups.append(sid)
train_groups = np.array(train_groups)

gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG.get('val_size', 0.2), random_state=CONFIG['seed'])
train_idx, val_idx = next(gss.split(train_images, groups=train_groups))

train_paths = train_images[train_idx]
val_paths = train_images[val_idx]

train_ds = SoilDataset(train_paths, ppm_dict, labels_df, train_transforms)
val_ds = SoilDataset(val_paths, ppm_dict, labels_df, valid_transforms)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'])
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

In [8]:
images, ppms, targets, sample_ids = next(iter(train_loader))

print("Images shape:", images.shape)
print("PPMs shape:", ppms.shape)
print("Targets shape:", targets.shape)
print("Sample IDs:", sample_ids)

print("Train samples:", len(train_ds))
print("Valid samples:", len(val_ds))

Images shape: torch.Size([16, 3, 384, 384])
PPMs shape: torch.Size([16, 1])
Targets shape: torch.Size([16, 11])
Sample IDs: ('H371', 'H616', 'H038', 'H126', 'H549', 'H374', 'H038', 'H372', 'H668', 'H031', 'G190', 'H615', 'H668', 'G190', 'H374', 'H405')
Train samples: 109
Valid samples: 26


In [9]:
class SoilGrainModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b4(pretrained=True)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        self.ppm_fc = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 32)
        )
        
        self.head = nn.Sequential(
            nn.Linear(in_features + 32, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 11)
        )

    def forward(self, x_img, x_ppm):
        img_feats = self.backbone(x_img)
        ppm_feats = self.ppm_fc(x_ppm)
        
        x = torch.cat((img_feats, ppm_feats), dim=1)
        logits = self.head(x)
        
        fractions = torch.softmax(logits, dim=1)
        cumulative = torch.cumsum(fractions, dim=1)
        
        return cumulative * 100.0


model = SoilGrainModel()
model = model.to(CONFIG['device'])
summary(model,input_data=(torch.randn(2, 3, CONFIG['img_size'], CONFIG['img_size']).to(CONFIG['device']),torch.randn(2, 1).to(CONFIG['device'])),depth=3,verbose=1)

Layer (type:depth-idx)                                       Output Shape              Param #
SoilGrainModel                                               [2, 11]                   --
├─EfficientNet: 1-1                                          [2, 1792]                 --
│    └─Sequential: 2-1                                       [2, 1792, 12, 12]         --
│    │    └─Conv2dNormActivation: 3-1                        [2, 48, 192, 192]         1,392
│    │    └─Sequential: 3-2                                  [2, 24, 192, 192]         4,146
│    │    └─Sequential: 3-3                                  [2, 32, 96, 96]           66,238
│    │    └─Sequential: 3-4                                  [2, 56, 48, 48]           197,586
│    │    └─Sequential: 3-5                                  [2, 112, 24, 24]          1,059,898
│    │    └─Sequential: 3-6                                  [2, 160, 24, 24]          2,306,724
│    │    └─Sequential: 3-7                                  [2, 2

Layer (type:depth-idx)                                       Output Shape              Param #
SoilGrainModel                                               [2, 11]                   --
├─EfficientNet: 1-1                                          [2, 1792]                 --
│    └─Sequential: 2-1                                       [2, 1792, 12, 12]         --
│    │    └─Conv2dNormActivation: 3-1                        [2, 48, 192, 192]         1,392
│    │    └─Sequential: 3-2                                  [2, 24, 192, 192]         4,146
│    │    └─Sequential: 3-3                                  [2, 32, 96, 96]           66,238
│    │    └─Sequential: 3-4                                  [2, 56, 48, 48]           197,586
│    │    └─Sequential: 3-5                                  [2, 112, 24, 24]          1,059,898
│    │    └─Sequential: 3-6                                  [2, 160, 24, 24]          2,306,724
│    │    └─Sequential: 3-7                                  [2, 2

In [10]:
class LogWeightedEMDLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.x = torch.tensor([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2.0, 6.3, 20.0, 63.0, 200.0])
        self.log_diffs = torch.log10(self.x[1:]) - torch.log10(self.x[:-1])
        
    def forward(self, preds, targets):
        self.log_diffs = self.log_diffs.to(preds.device)
        abs_diff = torch.abs(preds[:, :-1] - targets[:, :-1])
        emd = torch.sum(abs_diff * self.log_diffs, dim=1)
        return emd.mean()

In [11]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

def calculate_r2(preds, targets):
    # Exclude the final 200mm bin (index 10) to avoid divide-by-zero errors
    preds_np = preds[:, :-1].detach().cpu().numpy()
    targets_np = targets[:, :-1].detach().cpu().numpy()
    # R2 score can be negative for very bad predictions, clamp to 0 for readability if desired, 
    # but raw score is better for tracking.
    return r2_score(targets_np, preds_np)

model = SoilGrainModel().to(CONFIG['device'])
criterion = LogWeightedEMDLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=CONFIG['min_lr'])
scaler = GradScaler()
early_stopping = EarlyStopping(patience=4, min_delta=0.001)

best_loss = float('inf')
history = []

for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss = 0.0
    train_r2_sum = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{CONFIG['epochs']} [Train]", leave=False)
    for images, ppms, targets, _ in train_pbar:
        images = images.to(CONFIG['device'])
        ppms = ppms.to(CONFIG['device'])
        targets = targets.to(CONFIG['device'])
        
        optimizer.zero_grad()
        with autocast():
            preds = model(images, ppms)
            loss = criterion(preds, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        
        # Calculate R2 for the batch
        batch_r2 = calculate_r2(preds, targets)
        train_r2_sum += batch_r2
        
        train_pbar.set_postfix({'EMD': f"{loss.item():.4f}", 'R2': f"{batch_r2:.4f}"})
        
    scheduler.step()
    
    model.eval()
    val_loss = 0.0
    val_r2_sum = 0.0
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{CONFIG['epochs']} [Valid]", leave=False)
    with torch.no_grad():
        for images, ppms, targets, _ in val_pbar:
            images = images.to(CONFIG['device'])
            ppms = ppms.to(CONFIG['device'])
            targets = targets.to(CONFIG['device'])
            
            preds = model(images, ppms)
            loss = criterion(preds, targets)
            
            val_loss += loss.item()
            
            # Calculate R2 for the batch
            batch_r2 = calculate_r2(preds, targets)
            val_r2_sum += batch_r2
            
            val_pbar.set_postfix({'EMD': f"{loss.item():.4f}", 'R2': f"{batch_r2:.4f}"})
            
    avg_train_loss = train_loss / len(train_loader)
    avg_train_r2 = train_r2_sum / len(train_loader)
    
    avg_val_loss = val_loss / len(val_loader)
    avg_val_r2 = val_r2_sum / len(val_loader)
    
    is_best = ""
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), os.path.join(CONFIG['work_dir'], 'best_model.pth'))
        is_best = "★"
        
    history.append([
        epoch + 1, 
        f"{avg_train_loss:.4f}", 
        f"{avg_train_r2:.4f}",
        f"{avg_val_loss:.4f}", 
        f"{avg_val_r2:.4f}",
        f"{scheduler.get_last_lr()[0]:.2e}",
        is_best
    ])
    
    clear_output(wait=True)
    print("Target Metric: Logarithmically Weighted Earth Mover's Distance (EMD)\n")
    headers = ["Epoch", "Train Loss (EMD)", "Train R²", "Val Loss (EMD)", "Val R²", "LR", "Best"]
    print(tabulate(history, headers=headers, tablefmt="heavy_grid", stralign="center", numalign="center"))
    
    early_stopping(avg_val_loss)
    if early_stopping.early_stop:
        print(f"\nEarly stopping triggered at Epoch {epoch+1}. Restoring best model weights.")
        break

Target Metric: Logarithmically Weighted Earth Mover's Distance (EMD)

┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓
┃  Epoch  ┃  Train Loss (EMD)  ┃  Train R²  ┃  Val Loss (EMD)  ┃  Val R²  ┃    LR    ┃  Best  ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━╋━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━┫
┃    1    ┃       98.373       ┃  -4.3813   ┃      119.9       ┃ -1.5999  ┃ 4.97e-05 ┃   ★    ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━╋━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━┫
┃    2    ┃      90.9866       ┃  -7.4841   ┃     118.783      ┃ -1.5209  ┃ 4.88e-05 ┃   ★    ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━╋━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━┫
┃    3    ┃      83.8703       ┃  -4.9274   ┃     116.814      ┃  -1.442  ┃ 4.73e-05 ┃   ★    ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━╋━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━┫
┃    4    ┃      78.9405       ┃  -6.3428   ┃     114.094      ┃ -

In [12]:


BEST_MODEL = "/kaggle/working/best_model.pth"

sub_template = pd.read_csv(os.path.join(CONFIG['data_dir'], 'sample_submission.csv'))
valid_test_ids = sub_template['sample_id'].astype(str).values

all_files = glob.glob(os.path.join(CONFIG['data_dir'], '**/*.jpg'), recursive=True)
test_images = [p for p in all_files if 'test' in p.lower() or 'test' in os.path.basename(p).lower()]

class RobustTestDataset(Dataset):
    def __init__(self, image_paths, ppm_dict, valid_ids, transform=None):
        self.image_paths = image_paths
        self.ppm_dict = ppm_dict
        self.valid_ids = valid_ids
        self.transform = transform
        self.mean_ppm = sum(ppm_dict.values()) / len(ppm_dict) if ppm_dict else 1.0

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        filename = os.path.basename(img_path)
        camera_name = next((cam for cam in self.ppm_dict.keys() if cam in filename), None)
        ppm_val = self.ppm_dict[camera_name] if camera_name else self.mean_ppm
        ppm_tensor = torch.tensor([ppm_val], dtype=torch.float32)
        
        sample_id = next((tid for tid in self.valid_ids if tid in filename), None)
        if sample_id is None:
            sample_id = "UNKNOWN"
            
        return image, ppm_tensor, sample_id

test_ds = RobustTestDataset(test_images, ppm_dict, valid_test_ids, transform=valid_transforms)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

model = SoilGrainModel().to(CONFIG['device'])
model.load_state_dict(torch.load(BEST_MODEL, map_location=CONFIG['device']))
model.eval()

predictions = []
test_ids = []

test_pbar = tqdm(test_loader, desc="Generating Predictions")
with torch.no_grad():
    for images, ppms, ids in test_pbar:
        images = images.to(CONFIG['device'])
        ppms = ppms.to(CONFIG['device'])
        
        preds = model(images, ppms).cpu().numpy()
        predictions.extend(preds)
        test_ids.extend(ids)

pred_cols = [c for c in sub_template.columns if c != 'sample_id']
submission_df = pd.DataFrame(predictions, columns=pred_cols)
submission_df['sample_id'] = test_ids

submission_df = submission_df[submission_df['sample_id'] != "UNKNOWN"]

final_sub = submission_df.groupby('sample_id').mean().reset_index()

final_sub = pd.merge(sub_template[['sample_id']], final_sub, on='sample_id', how='left')

global_mean = final_sub[pred_cols].mean().fillna(50.0).values
for col in pred_cols:
    final_sub[col] = final_sub[col].fillna(sub_template[col])

final_sub[pred_cols[-1]] = 100.00

final_sub = final_sub[sub_template.columns]
final_sub.to_csv(os.path.join(CONFIG['work_dir'], 'submission.csv'), index=False)

Generating Predictions: 0it [00:00, ?it/s]